Se procederá la enriquecimiento del dataset de logistica 

- "Dataset_ALDIMI_Merged.csv" es el dataset destinado a la predicción de si es urgente reabastecer el stock o no ("Necesita_Reabastecimiento"), en base al consumo del paciente y el stock que se presenta en ese momento

In [41]:
import pandas as pd

#pegar ruta del dataset 
df = pd.read_csv("D:\Gitproyectos\Machine-learning\data\merged\Dataset_ALDIMI_Merged.csv")

print("Forma:", df.shape)

df["Punto_Reorden"] = df["Consumo_Diario"] * df["Lead_Time"]

df["Ratio_Stock"] = df["Stock_Actual"] / df["Punto_Reorden"]

def clasificar(fila):
    ratio = fila["Ratio_Stock"]

    if ratio > 5.6:
        return 0
    elif ratio > 2.2:
        return 1
    else:
        return 2

df["Necesita_Reabastecimiento"] = df.apply(clasificar, axis=1)

print("\nDistribución:")
print(df["Necesita_Reabastecimiento"].value_counts())

print("\nPorcentajes:")
print((df["Necesita_Reabastecimiento"].value_counts(normalize=True) * 100).round(2))

df.to_csv("Dataset_ALDIMI_Logistica_Enriquecido.csv", index=False)


<>:4: SyntaxWarning: invalid escape sequence '\G'
<>:4: SyntaxWarning: invalid escape sequence '\G'
C:\Users\mcabr\AppData\Local\Temp\ipykernel_12744\2148729990.py:4: SyntaxWarning: invalid escape sequence '\G'
  df = pd.read_csv("D:\Gitproyectos\Machine-learning\data\merged\Dataset_ALDIMI_Merged.csv")


Forma: (91250, 8)

Distribución:
Necesita_Reabastecimiento
1    40750
2    27921
0    22579
Name: count, dtype: int64

Porcentajes:
Necesita_Reabastecimiento
1    44.66
2    30.60
0    24.74
Name: proportion, dtype: float64


Se creó la columna Punto de Reorden a partir del Consumo Diario y el Lead Time porque permite estimar el nivel mínimo de inventario necesario antes de que ocurra un desabastecimiento, lo cual es un criterio estándar en gestión logística. A partir de esta variable se construyó el Ratio de Stock, que compara el inventario actual con el nivel crítico requerido, permitiendo medir de forma relativa la “holgura” o “riesgo” de stock en cada registro.

La variable objetivo Necesita_Reabastecimiento se definió en tres niveles utilizando umbrales sobre el Ratio de Stock para representar estados operativos interpretables: suficiente, alerta y crítico. Los valores utilizados (2.2 y 5.6) se eligieron a partir de los percentiles del propio dataset, lo que permite adaptar la clasificación a la distribución real de los datos en lugar de imponer reglas arbitrarias. Esto ayuda a reducir el desbalance extremo de clases y mejora la capacidad del modelo (como Random Forest o XGBoost) para aprender patrones útiles sin sesgarse hacia la clase dominante.

Se procederá la enriquecimiento del dataset de gravedad de pacientes

- "Dataset_ALDIMI_Merged_Clean.csv" es el dataset destinado a la predicción de la graveda 

D:\Gitproyectos\Machine-learning\data\processed\Dataset_ALDIMI_Merged_Clean.csv

In [42]:
import pandas as pd

df = pd.read_csv("D:\Gitproyectos\Machine-learning\data\processed\Dataset_ALDIMI_Merged_Clean.csv")

print("TIPOS DE VARIABLES:")
print(df.dtypes)

TIPOS DE VARIABLES:
Patient_ID                   int64
Cancer_Type                 object
Age                          int64
Gender                       int64
Smoking                      int64
Alcohol_Use                  int64
Obesity                      int64
Family_History               int64
Diet_Red_Meat                int64
Diet_Salted_Processed        int64
Fruit_Veg_Intake             int64
Physical_Activity            int64
Air_Pollution                int64
Occupational_Hazards         int64
BRCA_Mutation                int64
H_Pylori_Infection           int64
Calcium_Intake               int64
Overall_Risk_Score         float64
BMI                        float64
Physical_Activity_Level      int64
Risk_Level                  object
county_STATE                 int64
county_CTYNAME              object
county_POPESTIMATE2015     float64
dtype: object


<>:3: SyntaxWarning: invalid escape sequence '\G'
<>:3: SyntaxWarning: invalid escape sequence '\G'
C:\Users\mcabr\AppData\Local\Temp\ipykernel_12744\2096391599.py:3: SyntaxWarning: invalid escape sequence '\G'
  df = pd.read_csv("D:\Gitproyectos\Machine-learning\data\processed\Dataset_ALDIMI_Merged_Clean.csv")


In [44]:
df["Habitos_Riesgo"] = (
    df["Smoking"] +
    df["Alcohol_Use"] +
    df["Obesity"] +
    df["Air_Pollution"] +
    df["Occupational_Hazards"]
)

 
# 2. Riesgo clínico
 
df["Riesgo_Clinico"] = (
    df["Family_History"] +
    df["BRCA_Mutation"] +
    df["H_Pylori_Infection"]
)

 
# 3. Factor protector
 
df["Factor_Protector"] = (
    df["Fruit_Veg_Intake"] +
    df["Physical_Activity"] +
    df["Calcium_Intake"]
)

 
# 4. Balance global de riesgo
 
df["Balance_Riesgo"] = (
    df["Habitos_Riesgo"] +
    df["Riesgo_Clinico"] -
    df["Factor_Protector"]
)

 
# 5. Grupo de edad (segmentación clínica)
 
df["Edad_Rango"] = pd.cut(
    df["Age"],
    bins=[0, 30, 45, 60, 120],
    labels=["Joven", "Adulto", "Mayor", "Adulto_Mayor"]
)

 
# RESULTADO
 
print("Nuevas columnas creadas:")
print(df[[
    "Habitos_Riesgo",
    "Riesgo_Clinico",
    "Factor_Protector",
    "Balance_Riesgo",
    "Edad_Rango"
]].head())

# Guardar dataset enriquecido (opcional)
df.to_csv("Dataset_ALDIMI_GravedadPaciente_Enriquecido.csv", index=False)

Nuevas columnas creadas:
   Habitos_Riesgo  Riesgo_Clinico  Factor_Protector  Balance_Riesgo  \
0              41               0                18              23   
1              22               0                15               7   
2              13               0                13               0   
3              25               0                18               7   
4              30               1                19              12   

     Edad_Rango  
0         Mayor  
1  Adulto_Mayor  
2  Adulto_Mayor  
3  Adulto_Mayor  
4         Mayor  


In [45]:
# CANTIDAD POR CLASE
print("Cantidad de registros por clase:\n")
print(df["Risk_Level"].value_counts())

# PORCENTAJE POR CLASE
print("\nPorcentaje de registros por clase:\n")
print(df["Risk_Level"].value_counts(normalize=True) * 100)

# TABLA RESUMEN (BONITO PARA INFORME)
resumen = pd.DataFrame({
    "Cantidad": df["Risk_Level"].value_counts(),
    "Porcentaje": df["Risk_Level"].value_counts(normalize=True) * 100
})

print("\nResumen completo:\n")
print(resumen)

Cantidad de registros por clase:

Risk_Level
Medium    2368
Low        484
High       148
Name: count, dtype: int64

Porcentaje de registros por clase:

Risk_Level
Medium    78.933333
Low       16.133333
High       4.933333
Name: proportion, dtype: float64

Resumen completo:

            Cantidad  Porcentaje
Risk_Level                      
Medium          2368   78.933333
Low              484   16.133333
High             148    4.933333


In [47]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE

# SEPARAR VARIABLES
X = df.drop("Risk_Level", axis=1)
y = df["Risk_Level"]

# IDENTIFICAR COLUMNAS
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object"]).columns

# PREPROCESAMIENTO
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ]
)

# Transformar X
X_processed = preprocessor.fit_transform(X)

# TRAIN / TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# APLICAR SMOTE SOLO EN TRAIN
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

# RESULTADOS
print("ANTES DEL BALANCEO:")
print(y_train.value_counts())

print("\nDESPUÉS DEL BALANCEO:")
print(pd.Series(y_train_bal).value_counts())

ANTES DEL BALANCEO:
Risk_Level
Medium    1895
Low        387
High       118
Name: count, dtype: int64

DESPUÉS DEL BALANCEO:
Risk_Level
Medium    1895
High      1895
Low       1895
Name: count, dtype: int64


Se realizó un proceso de enriquecimiento y transformación de datos con el objetivo de mejorar la capacidad predictiva del modelo orientado a la clasificación del nivel de prioridad de atención (Risk_Level: Bajo, Medio y Alto). Este proceso consistió en la creación de nuevas variables derivadas a partir de las características originales del dataset, agrupando factores relacionados como hábitos de riesgo, condiciones clínicas y factores protectores. Estas nuevas variables permiten resumir información dispersa en indicadores compuestos más representativos del estado general del paciente, facilitando que el modelo identifique patrones no lineales de mayor complejidad.

Este enriquecimiento es clave porque mejora la calidad de la señal de los datos, reduce la fragmentación de variables individuales y permite capturar interacciones entre factores que por separado no serían evidentes. Como resultado, el modelo puede diferenciar con mayor precisión entre pacientes de bajo, medio y alto riesgo, especialmente en casos críticos donde múltiples factores clínicos y de estilo de vida influyen simultáneamente en la clasificación final.